# BigQuery Generative AI & ML 분석 실습

이 노트북은 BigQuery에서 원격 모델(Remote Model)을 생성하고 연동하여, 대규모 이커머스 데이터에 대해 고급 LLM 텍스트 생성, 다국어 처리 및 벡터 검색(Vector Search) 엔진을 구축하는 실습을 진행합니다.

### 학습 목표
1. **`ML.GENERATE_TEXT`**: SQL 내에서 Gemini 모델을 호출해 이탈 방지용 맞춤형 마케팅 메일을 자동으로 작성합니다.
2. **정형 데이터 파싱**: LLM의 응답 형식을 JSON으로 제어하고, BigQuery 내에서 `JSON_VALUE` 및 `JSON_QUERY` 함수로 필요한 속성을 파싱합니다.
3. **`ML.GENERATE_EMBEDDING`**: 상품 설명 텍스트를 고차원 밀집 벡터(Embedding) 데이터로 대량 변환하여 적재합니다.
4. **`VECTOR_SEARCH`**: 적재된 벡터 테이블과 자연어 질의 벡터 간의 유사도 연산을 실행하여 완전한 SQL 기반의 벡터 매칭 검색 엔진을 구현합니다.

### 작업 파이프라인
1. **환경 설정 및 원격 모델 설정**: GCP Vertex AI 플랫폼과 연동할 Remote Connection을 생성하고 Gemini 및 텍스트 임베딩 모델 인스턴스 등록
2. **마케팅 메일 자동 생성**: VIP 고객 중 이탈 위험 고객을 타겟으로 `ML.GENERATE_TEXT`를 사용하여 맞춤형 이메일 생성
3. **다국어 번역 및 태그 추출**: 상품명을 한글로 번역하고 의미론적 검색 키워드를 JSON 구조로 정형화하여 추출
4. **상품 텍스트 임베딩 생성**: 상품 정보 텍스트를 고차원 임베딩 벡터로 일괄 변환하여 임베딩 테이블 구축
5. **자연어 기반 벡터 검색**: 사용자의 자연어 입력 쿼리를 벡터 변환 후 `VECTOR_SEARCH`를 사용하여 코사인 유사도가 높은 관련 상품 조회

## Step 1: 환경 초기화 및 원격 모델 설정

BigQuery에 저장된 데이터에서 직접 머신러닝 및 거대 생성 모델(Gemini)을 즉시 기동하기 위해, BigQuery 원격 연결(Remote Connection)을 수립하고 Vertex AI 서비스에 연결되는 Gemini 모델 리소스를 생성 및 등록합니다.

In [ ]:
# [Qwiklabs 환경 대응] 만약 BigQuery API 호출 또는 매직 실행 중 HTTP Error 401 (Invalid authentication credentials) 에러가 발생한다면,
# 아래 줄의 주석을 해제하고 실행하여 1회성 수동 인증(ADC 생성)을 수행해 주세요.
# !gcloud auth application-default login --no-launch-browser --quiet

import google.auth
from google.cloud import bigquery

# active GCP project ID 조회 및 클라이언트 구성
_, PROJECT_ID = google.auth.default()
bq_client = bigquery.Client(project=PROJECT_ID)
print(f"Active Project: {PROJECT_ID}")

In [ ]:
# 1. Gemini Text 및 Embedding 모델 생성 (Python 클라이언트를 통해 동적 프로젝트 ID 바인딩)
create_models_query = f"""
-- 1. Gemini Text 모델 생성
CREATE OR REPLACE MODEL thelook_ecommerce.gemini_flash
REMOTE WITH CONNECTION `{PROJECT_ID}.us-central1.vertex-connection`
OPTIONS(ENDPOINT = 'gemini-2.5-flash');

-- 2. Text Embedding 모델 생성
CREATE OR REPLACE MODEL thelook_ecommerce.text_embedding
REMOTE WITH CONNECTION `{PROJECT_ID}.us-central1.vertex-connection`
OPTIONS(ENDPOINT = 'text-embedding-004');
"""

print("원격 모델 생성 쿼리 실행 중...")
query_job = bq_client.query(create_models_query)
query_job.result()  # 완료 대기
print("원격 모델(gemini_flash, text_embedding) 생성 완료!")

## Step 2: 이탈 방지 개인화 추천 메일 생성 (`ML.GENERATE_TEXT`)

이탈 가능성이 높다고 판단된 고객군(예: 3개월간 구매가 없었으나 이전 구매 금액이 높은 VIP 고객) 중 일부를 선별하여, 그들이 마지막으로 구매했던 브랜드와 카테고리를 활용해 **개인화된 할인 쿠폰 마케팅 이메일**을 대량 생성합니다.

BigQuery SQL 쿼리 내부에서 `ML.GENERATE_TEXT`와 Gemini 모델을 사용하여, 각 고객 맞춤 정보를 프롬프트로 동적 조립한 후 이메일 제목과 본문을 자동으로 추출해냅니다.

In [ ]:
%%bigquery
-- 이탈 위기 VIP 고객 대상 개인화 마케팅 메일 생성 (샘플 고객 선별)
WITH target_users AS (
  SELECT 
    u.id AS user_id,
    u.first_name,
    u.email,
    -- 최근 구매한 상품 정보
    p.brand AS last_bought_brand,
    p.category AS last_bought_category,
    p.name AS last_bought_product
  FROM thelook_ecommerce.users u
  JOIN thelook_ecommerce.order_items oi ON u.id = oi.user_id
  JOIN thelook_ecommerce.products p ON oi.product_id = p.id
  WHERE u.id IN (367, 100, 500, 1000) -- 고유 고객 샘플 선별
  QUALIFY ROW_NUMBER() OVER(PARTITION BY u.id ORDER BY oi.created_at DESC) = 1
),
prompts AS (
  SELECT
    user_id,
    first_name,
    email,
    CONCAT(
      "당신은 패션 쇼핑몰 'TheLook'의 친근한 마케팅 담당자입니다. 최근 구매 이력이 없는 고객을 다시 유치하기 위한 개인화된 할인 프로모션 이메일을 한국어로 작성해주세요. ",
      "고객 이름: ", first_name, ". ",
      "마지막으로 구매한 상품: 브랜드 '", last_bought_brand, "', 카테고리 '", last_bought_category, "', 상품명 '", last_bought_product, "'. ",
      "이 고객에게 동일한 브랜드 혹은 카테고리의 상품에 사용할 수 있는 20% 할인 쿠폰(코드: MISSYOU20)을 제안하며 구매를 유도하는 매력적인 메일을 써주세요. ",
      "제목은 반드시 '[TheLook]'으로 시작해야 하며, 본문은 정중하고 따뜻하며 간결한 어조로 작성해주세요. 이름이나 상품명 등에 플레이스홀더([Name] 등)를 남기지 말고 완성된 텍스트로 출력해주세요."
    ) AS prompt
  FROM target_users
)
SELECT 
  user_id,
  email,
  ml_generate_text_llm_result AS generated_email
FROM ML.GENERATE_TEXT(
  MODEL thelook_ecommerce.gemini_flash,
  (SELECT * FROM prompts),
  STRUCT(0.3 AS temperature, 1000 AS max_output_tokens, TRUE AS flatten_json_output)
);

## Step 3: 다국어 번역 및 검색 키워드 태그 추출 (JSON Mode)

TheLook 쇼핑몰의 상품 이름(`product_name`)은 영어로 등록되어 있습니다. 한국 고객층을 위한 **한국어 상품명 번역**을 일괄 수행하고, 자연어 검색 인덱스로 사용할 수 있도록 **상품 특징 키워드 태그(Tags)를 JSON 포맷으로 자동 추출**합니다.

Gemini의 **JSON Mode**를 강제하여 출력 데이터의 형식을 정형화함으로써, 결과값을 BigQuery 내에서 파싱하여 바로 컬럼으로 가공할 수 있게 합니다.

In [ ]:
%%bigquery
-- 상품명 한국어 번역 및 검색 키워드 태그 추출 (Prompt-based JSON)
WITH target_products AS (
  SELECT 
    id AS product_id,
    name AS english_name,
    brand,
    category,
    retail_price
  FROM thelook_ecommerce.products
  WHERE id IN (19666, 18458, 21572, 22000) -- 일부 상품 선별
),
prompts AS (
  SELECT
    product_id,
    english_name,
    CONCAT(
      "제시된 상품 정보를 분석하여 상품명을 자연스러운 한국어로 번역하고, 상품의 스타일이나 소재를 나타내는 검색용 키워드 태그 3개를 추출해주세요. ",
      "영어 상품명: '", english_name, "', 브랜드: '", brand, "', 카테고리: '", category, "'. ",
      "출력은 반드시 'korean_name'(번역된 한국어 상품명 문자열)과 'tags'(키워드 태그 문자열 배열, 예: ['면', '캐주얼', '슬림핏']) 두 필드를 가진 하나의 JSON 객체여야 합니다. ",
      "마크다운 코드 블록 등의 형식을 사용하지 말고, 오직 순수한 JSON 문자열만 반환해주세요."
    ) AS prompt
  FROM target_products
)
SELECT 
  product_id,
  english_name,
  -- JSON 파싱하여 개별 컬럼으로 추출
  JSON_VALUE(ml_generate_text_llm_result, '$.korean_name') AS korean_name,
  JSON_QUERY(ml_generate_text_llm_result, '$.tags') AS search_tags,
  ml_generate_text_llm_result AS raw_json
FROM ML.GENERATE_TEXT(
  MODEL thelook_ecommerce.gemini_flash,
  (SELECT * FROM prompts),
  STRUCT(
    0.2 AS temperature,
    TRUE AS flatten_json_output
  )
);

## Step 4: 상품 텍스트 임베딩 생성 (`ML.GENERATE_EMBEDDING`) 및 자연어 쿼리 기반 벡터 검색 (`VECTOR_SEARCH`)

이메일이나 키워드 검색 외에도, 고객이 입력한 자연어 쿼리("남자용 정장 바지", "comfy summer t-shirt")와 의미론적으로 가장 유사한 상품을 매칭하는 **벡터 검색(Vector Search)** 엔진을 구축합니다.

1.  `ML.GENERATE_EMBEDDING`을 활용하여 상품명과 브랜드, 카테고리를 합친 상품 설명 텍스트의 768차원 벡터 임베딩을 생성하여 테이블로 저장합니다.
2.  `VECTOR_SEARCH` 함수를 사용하여 임의의 검색 쿼리 벡터와 가장 코사인 유사도가 높은 상품 Top 5를 색인을 활용해 고속으로 조회합니다.

In [ ]:
%%bigquery
-- 1. 상품 카탈로그 임베딩 테이블 생성 (테스트용 100개 샘플링)
CREATE OR REPLACE TABLE thelook_ecommerce.product_embeddings AS
SELECT 
  id AS product_id,
  name AS product_name,
  brand,
  category,
  ml_generate_embedding_result AS product_embedding -- 768차원 벡터값 추출
FROM ML.GENERATE_EMBEDDING(
  MODEL thelook_ecommerce.text_embedding,
  (
    SELECT 
      id,
      name,
      brand,
      category,
      CONCAT("상품명: ", name, ", 브랜드: ", brand, ", 카테고리: ", category) AS content
    FROM thelook_ecommerce.products
    LIMIT 100
  ),
  STRUCT(TRUE AS flatten_json_output)
);

In [ ]:
%%bigquery
-- 2. 자연어 검색어 벡터를 사용한 상품 유사도 검색 (Vector Search)
WITH query_embedding AS (
  SELECT ml_generate_embedding_result AS q_embedding
  FROM ML.GENERATE_EMBEDDING(
    MODEL thelook_ecommerce.text_embedding,
    (SELECT '여름용 편안한 면 티셔츠' AS content),
    STRUCT(TRUE AS flatten_json_output)
  )
)
SELECT 
  base.product_id,
  base.product_name,
  base.brand,
  base.category,
  ROUND(distance, 4) AS cosine_distance
FROM VECTOR_SEARCH(
  TABLE thelook_ecommerce.product_embeddings,
  'product_embedding',
  TABLE query_embedding,
  'q_embedding',
  top_k => 5,
  distance_type => 'COSINE'
);